# Tutorial 14: Custom PyTorch model (`%%krauncher`)

Runs a hand-written CNN training cell on the cheapest suitable remote GPU. The
notebook kernel stays local; only the marked cell goes remote. The model is not
in the HF registry, exercising the analyzer's fallback path.

Config (API key, broker/analyzer URLs) is loaded by the SDK from the sibling
`cas-client/.env` via `CAS_CLIENT_CONFIG` — no secrets in the notebook. The
first `%%krauncher` cell uses `--estimate` to classify and quote only; the
second runs it for real and injects the outputs back as notebook variables.

In [ ]:
%env CAS_CLIENT_CONFIG=../../cas-client/.env

In [ ]:
%load_ext krauncher_magic

In [ ]:
epochs = 5
batch_size = 32

In [ ]:
%%krauncher --in epochs,batch_size --out initial_loss,final_loss,device --pip torch --timeout 300 --estimate
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.act(self.bn(self.conv(x))))

class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    n_batches = 50
    for i in range(n_batches):
        images = torch.randn(batch_size, 3, 64, 64, device=device)
        labels = torch.randint(0, 10, (batch_size,), device=device)

        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        print(f"\rEpoch {epoch}/{epochs}  batch {i+1}/{n_batches}  loss={loss.item():.4f}", end="")

    avg = epoch_loss / n_batches
    losses.append(avg)
    print(f"\rEpoch {epoch}/{epochs}  avg_loss={avg:.4f}          ")

initial_loss = round(losses[0], 4)
final_loss = round(losses[-1], 4)
device = str(device)

In [ ]:
%%krauncher --in epochs,batch_size --out initial_loss,final_loss,device --pip torch --timeout 300
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.act(self.bn(self.conv(x))))

class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    n_batches = 50
    for i in range(n_batches):
        images = torch.randn(batch_size, 3, 64, 64, device=device)
        labels = torch.randint(0, 10, (batch_size,), device=device)

        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        print(f"\rEpoch {epoch}/{epochs}  batch {i+1}/{n_batches}  loss={loss.item():.4f}", end="")

    avg = epoch_loss / n_batches
    losses.append(avg)
    print(f"\rEpoch {epoch}/{epochs}  avg_loss={avg:.4f}          ")

initial_loss = round(losses[0], 4)
final_loss = round(losses[-1], 4)
device = str(device)

In [ ]:
print(f"initial_loss={initial_loss}  final_loss={final_loss}  device={device}")